In [1]:
from datetime import datetime, timedelta
from utils.spark_session import get_sparkSession 
from utils.utilities import generate_date
from utils.dq_checks import dq_validate_checks

In [2]:
from pyspark.sql.functions import current_timestamp, current_date, lit, expr, to_date, when, lower, upper, trim, concat_ws, xxhash64, cast, col, lead, date_sub, coalesce, date_format
from pyspark.sql.window import Window

In [3]:
spark = get_sparkSession("Load gold.fact_ads_performance")

In [4]:
print("SPARK-APP : spark-UI - " + spark.sparkContext.uiWebUrl)

SPARK-APP : spark-UI - http://ba7b2bbb9aec:4040


In [5]:
from utils.load_controller import insert_control_logs, get_max_loadTimestamp
from delta import DeltaTable

In [6]:
_schema_name = "gold"
_table_name = "fact_ads_performance"
_fullname = f"{_schema_name}.{_table_name}"
_script_name = "gold_fact_ads_performance.ipynb"
_silver_table = "silver.fact_ads_performance"

print(f"SPARK-APP : Loading started for {_fullname}")

SPARK-APP : Loading started for gold.fact_ads_performance


In [7]:
#spark config

spark.conf.set("spark.sql.shuffle.partitions",16)


In [8]:
#Reading from silver.customer and generating surrogate key and business key

df = spark.read.table(_silver_table)

print(f"SPARK-APP: Silver Table Data count : {df.count()}")
df.show(truncate = False)

SPARK-APP: Silver Table Data count : 1
+-----------+----------+------+-------+---------------+-----------+------+------+--------+----------------+-----------------+--------------------------+---------------------------------+--------------------------+---------------------------------+------------------------+
|campaign_id|ads_date  |store |channel|campaign_name  |impressions|clicks|cost  |currency|orders_generated|revenue_generated|created_on                |created_by                       |modified_on               |modified_by                      |source_file             |
+-----------+----------+------+-------+---------------+-----------+------+------+--------+----------------+-----------------+--------------------------+---------------------------------+--------------------------+---------------------------------+------------------------+
|AD1001     |2025-01-01|AMZ_US|MKT    |Amazon Campaign|1000       |50    |200.00|USD     |5               |2250.00          |2026-01-29 00:56:

In [9]:
# DQ validations

_row = df \
        .select("source_file") \
        .distinct() \
        .limit(1) \
        .first()

_source_file = "UNKNOWN" if _row is None else _row[0]

df = dq_validate_checks(df, spark, _schema_name, _table_name, _source_file)

print("SPARK-APP: DQ validations completed")
print(f"SPARK-APP: Table Data count after DQ validations : {df.count()}")

SPARK-APP: DQ validations completed
SPARK-APP: Table Data count after DQ validations : 1


In [10]:
#Adding audit columns

df = df.withColumn("created_on", current_timestamp()) \
       .withColumn("created_by", lit(_script_name)) \
       .withColumn("modified_on", current_timestamp()) \
       .withColumn("modified_by", lit(_script_name))

print(f"SPARK-APP: Silver Table Data count : {df.count()}")

# Generating surrogate key and business key

df_sk = df.withColumn("ads_sk", xxhash64(concat_ws("||", "campaign_id", "ads_date", "store", "channel")).cast("bigint"))

df_sk.show(truncate = False)

SPARK-APP: Silver Table Data count : 1
+-----------+----------+------+-------+---------------+-----------+------+------+--------+----------------+-----------------+--------------------------+-------------------------------+--------------------------+-------------------------------+------------------------+--------------------+
|campaign_id|ads_date  |store |channel|campaign_name  |impressions|clicks|cost  |currency|orders_generated|revenue_generated|created_on                |created_by                     |modified_on               |modified_by                    |source_file             |ads_sk              |
+-----------+----------+------+-------+---------------+-----------+------+------+--------+----------------+-----------------+--------------------------+-------------------------------+--------------------------+-------------------------------+------------------------+--------------------+
|AD1001     |2025-01-01|AMZ_US|MKT    |Amazon Campaign|1000       |50    |200.00|USD     |5

In [11]:
# Join with other dimension tables to get their surrogate keys

df_gold_store = spark.read.table("gold.dim_store").select("store_sk", "store_code")
df_gold_channel = spark.read.table("gold.dim_channel").select("channel_sk", "channel_code")
df_ads_date = spark.read.table("gold.dim_date").select("date_sk","date")
df_gold_currency = spark.read.table("gold.dim_currency").select("currency_sk", "currency_code")

df_silver = df_sk.join(df_gold_store, how="inner", on=df_sk.store==df_gold_store.store_code) \
                 .join(df_gold_channel, how="inner", on=df_sk.channel==df_gold_channel.channel_code) \
                 .join(df_gold_currency, how = "inner", on=df_sk.currency==df_gold_currency.currency_code) \
                 .join(df_ads_date, how = "left", on=df_sk.ads_date==df_ads_date.date) \
                 .select("ads_sk", "campaign_id", coalesce(df_ads_date.date_sk,date_format(df_sk.ads_date, "yyyyMMdd").cast("int")).alias("ads_date"),
                         "store_sk", "channel_sk", "currency_sk", "campaign_name", "impressions", "clicks", "cost", "orders_generated", "revenue_generated",
                         "source_file", "created_on", "created_by", "modified_on", "modified_by")

df_silver.show(truncate = False)

+--------------------+-----------+--------+--------+----------+-----------+---------------+-----------+------+------+----------------+-----------------+------------------------+--------------------------+-------------------------------+--------------------------+-------------------------------+
|ads_sk              |campaign_id|ads_date|store_sk|channel_sk|currency_sk|campaign_name  |impressions|clicks|cost  |orders_generated|revenue_generated|source_file             |created_on                |created_by                     |modified_on               |modified_by                    |
+--------------------+-----------+--------+--------+----------+-----------+---------------+-----------+------+------+----------------+-----------------+------------------------+--------------------------+-------------------------------+--------------------------+-------------------------------+
|-6155788525161618596|AD1001     |20250101|2       |1         |1          |Amazon Campaign|1000       |50    |20

In [12]:
# Truncate table for full load
dt_fact_ads_performance = DeltaTable.forName(spark, f"{_schema_name}.{_table_name}")

if get_max_loadTimestamp(spark, _schema_name, _table_name) == '1900-01-01 00:00:00.000000':
    
    #Full-load so truncate dim table
    spark.sql("SET spark.databricks.delta.retentionDurationCheck.enabled=false")
    
    dt_fact_ads_performance.delete()
    dt_fact_ads_performance.vacuum(0)


In [13]:
# Merge
dt_fact_ads_performance.alias("t").merge(
    df_silver.alias("s"),
    "t.ads_sk = s.ads_sk"
).whenMatchedUpdate(set = {   
    "campaign_name":"s.campaign_name",
    "impressions":"s.impressions",
    "clicks":"s.clicks",
    "cost":"s.cost",
    "orders_generated":"s.orders_generated",
    "revenue_generated":"s.revenue_generated",
    "source_file" : "s.source_file",
    "modified_on" : "s.modified_on",
    "modified_by" : "s.modified_by"
    }).whenNotMatchedInsertAll().execute()
                     
print("SPARK-APP: Merge completed") 

SPARK-APP: Merge completed


In [14]:
hist = dt_fact_ads_performance.history().limit(1).select("version", "operationMetrics.executionTimeMs",
                                          "operationMetrics.numTargetRowsInserted", "operationMetrics.numTargetRowsUpdated",
                                          "operationMetrics.numOutputRows")

hist.show()
row = hist.first()

loaded_count = int("0" if row is None else row["numTargetRowsInserted"]) + int("0" if row is None else row["numTargetRowsUpdated"])


+-------+---------------+---------------------+--------------------+-------------+
|version|executionTimeMs|numTargetRowsInserted|numTargetRowsUpdated|numOutputRows|
+-------+---------------+---------------------+--------------------+-------------+
|      1|           6322|                    1|                   0|            1|
+-------+---------------+---------------------+--------------------+-------------+



In [15]:
spark.read.table(f"{_schema_name}.{_table_name}").show()

+--------------------+-----------+--------+--------+----------+-----------+---------------+-----------+------+------+----------------+-----------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|              ads_sk|campaign_id|ads_date|store_sk|channel_sk|currency_sk|  campaign_name|impressions|clicks|  cost|orders_generated|revenue_generated|          created_on|          created_by|         modified_on|         modified_by|         source_file|
+--------------------+-----------+--------+--------+----------+-----------+---------------+-----------+------+------+----------------+-----------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|-6155788525161618596|     AD1001|20250101|       2|         1|          1|Amazon Campaign|       1000|    50|200.00|               5|          2250.00|2026-01-29 03:47:...|gold_fact_ads_per...|2026-01-29 03:47:...|gold_fact_a

In [16]:
#Writing log data to load_controller

_data = []

_data.append([_schema_name, _schema_name, _table_name, "delta-load", "merge", str(datetime.now()), "success", loaded_count, _script_name, _script_name])

insert_control_logs(spark, _data)
    
print(f"SPARK-APP: Data written to load_controller")

SPARK-APP: Data written to load_controller


In [17]:
spark.sql(f"select * from gold.load_controller where schema_name = '{_schema_name}' and table_name = '{_table_name}' order by created_on desc").show()

+-----+-----------+--------------------+----------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+
|layer|schema_name|          table_name| load_type|write_mode|      load_timestamp|load_status|loaded_count|          created_on|          created_by|         modified_on|         modified_by|
+-----+-----------+--------------------+----------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+
| gold|       gold|fact_ads_performance|delta-load|     merge|2026-01-29 03:48:...|    success|           1|2026-01-29 03:48:...|gold_fact_ads_per...|2026-01-29 03:48:...|gold_fact_ads_per...|
+-----+-----------+--------------------+----------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+



In [18]:
#Generating symlink manifest file

dt = DeltaTable.forName(spark, f"{_schema_name}.{_table_name}")
dt.generate("symlink_format_manifest")

print("SPARK-APP: symlink manifest file generated")

SPARK-APP: symlink manifest file generated


In [19]:
spark.stop()